In [ ]:
import pandas as pd
import numpy as np

# load spotify data
df = pd.read_csv(r'excel_data\Alltime_Spotify_Streaming_Data.csv')

print(f"Loaded Database: {df.shape[0]} rows, {df.shape[1]} columns")


In [ ]:
df

In [ ]:
############################
# STEP 1: HEADERS CLEANING #
############################

df.columns = df.columns.str.replace('_', ' ')

df = df.rename(columns={
    'track album name': 'track artist',
    'track album name2': 'track album name'
})

print(df.columns)


In [ ]:
#############################
# STEP 2: CATEGORICAL TYPOS #
#############################

cleanup_map = {
    'Android OS 5.1.1 API 22 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'iOS 9.2.1 (iPad2,2)': 'iPad2',
    'Android OS 6.0.1 API 23 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android OS 7.0 API 24 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android-tablet OS 7.0 API 24 (samsung, SAMSUNG-SM-G920A)': 'SAMSUNG-SM-G920A',
    'Android OS 6.0.1 API 23 (HUAWEI, KIW-L24)': 'KIW-L24',
    'Android OS 8.0.0 API 26 (motorola, moto g(6) play)': 'moto g(6) play',
    'Android OS 9 API 28 (motorola, moto g(6) play)': 'moto g(6) play',
    'Android OS 8.0.0 API 26 (samsung, SM-G950U)': 'SM-G950U',
    'Android OS 9 API 28 (samsung, SM-G950U)': 'SM-G950U',
    'Android OS 11 API 30 (samsung, SM-G781U)': 'SM-G781U',
    'Android OS 12 API 31 (samsung, SM-G781U)': 'SM-G781U',
    'not_applicable': np.nan # handle ghost values     
}

df['streaming platform'] = df['streaming platform'].replace(cleanup_map)

print(df['streaming platform'].unique())

In [ ]:
###############################
# STEP 3: DATE TIME DATA TYPE #
###############################

print(df['time stamp'].dtype)

df['time stamp'] = pd.to_datetime(df['time stamp'],errors = 'coerce')

df['time stamp']

In [ ]:
###################################
# STEP 4: LOGICAL INTEGRITY CHECK #
#      A: (No Start or End)       #
###################################

impossible_track1 = df['reason start'].isna()
impossible_track2 = df['reason end'].isna()

print(df.loc[impossible_track1])
print(df.loc[impossible_track2])

In [ ]:
###################################
# STEP 4: LOGICAL INTEGRITY CHECK #
#      b: (less Than 0 ms Played) #
###################################
# step 4b: check logical integrity (less than 0 ms played)
impossible_track3 = df['ms played'] < 0

print(df.loc[impossible_track3])

In [ ]:
#######################################
# STEP 5: REMOVE UNNECESSARY ROWS     #
#      A: (Tracks that are not songs) #
#######################################

# check if there are podcast or audiobook tracks
not_songs = df['episode name'].notna() | df['audiobook title'].notna()
print(df.loc[not_songs])

# remove these tracks from dataframe
df = df[~not_songs]

In [ ]:
#######################################
# STEP 5: REMOVE UNNECESSARY ROWS     #
#      B: (Tracks played for 0 ms)    #
#######################################

# check if there are tracks played for 0 seconds
zero_ms = df['ms played'] == 0
print(df.loc[zero_ms])

# remove these tracks from dataframe
df = df[~zero_ms]

In [ ]:
######################################
# STEP 6: REMOVE UNNECESSARY COLUMNS #
######################################

df = df.drop(['episode name', 'episode show name', 'episode spotify url', 
              'audiobook title', 'audiobook url', 'audiobook chapter url',
              'audiobook chapter title', 'offline timestamp'],axis=1)

df

In [ ]:
######################################
# STEP 7: INDENTIFY UNIQUE TRACKS    #
######################################

df_unique = df.drop_duplicates(subset=['track spotify url'])
df_unique = df_unique[['track title', 'track artist', 'track spotify url']]

# determine number of unique songs
num_unique_songs = df_unique.shape[0]
user_message = f"You have listened to {num_unique_songs} different tracks in the last 10 years!"
print(user_message)

# save list of unique songs as csv file
df_unique.to_csv('unique_songs.csv', index=False)